# 04 — Visualisation

This notebook covers all visualisation functions in alsdb:

- Point cloud: 4-panel overview, individual 2-D panels, 3-D scatter
- GEDI waveform plots
- Gridded products from an `ALSZarrStore` (CHM, DTM, DSM, gap, LAI, biomass, metrics)

**Paths** — replace `/path/to/...` with your actual paths.

## Setup

In [ ]:
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

from alsdb import ALSProvider
from alsdb.storage import ALSZarrStore

reader = ALSProvider(storage_type="local", uri="/path/to/my_array")
store  = ALSZarrStore("/path/to/forest.zarr")

BBOX = (308_000, 4_688_000, 310_000, 4_690_000)
YEAR = 2021

# Query a point-cloud tile for the 2-D plot functions
df = reader.query_bbox(*BBOX, year=YEAR)

## 4-panel overview

`plot_overview` produces a 2×2 figure: DSM + hillshade, RGB orthoimage, intensity, and classification map.  
All 2-D plot functions rasterise the point cloud on the fly, so they stay fast even for multi-million-point tiles.

In [ ]:
from alsdb.utils.viz import plot_overview

fig = plot_overview(df, resolution=1.0)
fig.savefig("overview.png", dpi=150, bbox_inches="tight")

## Individual 2-D panels

Each panel function accepts an optional `ax` argument so you can compose your own layout.

In [ ]:
import matplotlib.pyplot as plt
from alsdb.utils.viz import plot_dsm, plot_rgb, plot_intensity, plot_classification

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_dsm(df, resolution=1.0, hillshade=True, ax=axes[0])
plot_rgb(df, resolution=1.0, ax=axes[1])
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_intensity(df, resolution=1.0, ax=axes[0])
plot_classification(df, resolution=1.0, ax=axes[1])
fig.tight_layout()

## 3-D point cloud

`plot_pointcloud_3d` supports `backend="matplotlib"` (static) or `backend="plotly"` (interactive).  
Colour can be `"Z"`, `"RGB"`, `"Intensity"`, or `"Classification"`.

In [ ]:
from alsdb.utils.viz import plot_pointcloud_3d

# Static — coloured by elevation
plot_pointcloud_3d(df, color_by="Z", backend="matplotlib")

# Interactive — coloured by RGB (requires plotly)
# plot_pointcloud_3d(df, color_by="RGB", backend="plotly", max_points=200_000)

## GEDI waveform plots

See notebook `03_waveform.ipynb` for how to generate a `result` object.

In [ ]:
from alsdb.processing.waveform import simulate_waveform
from alsdb.utils.viz import plot_waveform, plot_rh_profile

result = simulate_waveform(
    provider=reader,
    center_x=308_500.0,
    center_y=4_689_000.0,
    footprint_radius=12.5,
    year=YEAR,
)

# Raw waveform vs elevation with annotated RH levels
fig = plot_waveform(result)
fig.savefig("waveform.png", dpi=150, bbox_inches="tight")

In [ ]:
# GEDI L2A style: RH(p) curve + W(h) energy density with layer peaks annotated
fig = plot_rh_profile(result)
fig.savefig("rh_profile.png", dpi=150, bbox_inches="tight")

## Gridded products from ALSZarrStore

All raster plot functions read from an `ALSZarrStore` at a given resolution and year.  
If the store contains only one survey year, the `year=` argument can be omitted.

In [ ]:
from alsdb.utils.viz_raster import (
    plot_chm, plot_dtm, plot_dsm as plot_dsm_raster,
    plot_agb, plot_gap, plot_lai,
    plot_metrics,
    plot_products, plot_products_agb,
)

# Individual panels
plot_chm(store, resolution=1.0,  year=YEAR)
plot_dtm(store, resolution=1.0,  year=YEAR, hillshade=True)
plot_dsm_raster(store, resolution=1.0,  year=YEAR)

In [ ]:
plot_agb(store, resolution=10.0, year=YEAR)
plot_gap(store, resolution=10.0, year=YEAR)
plot_lai(store, resolution=10.0, year=YEAR)

In [ ]:
# All six structural metrics in one figure (h50, h75, h95, hmean, cc, density)
plot_metrics(store, resolution=10.0, year=YEAR)

In [ ]:
# Three-panel overview: DTM | DSM | CHM
plot_products(store, resolution=1.0, year=YEAR)

In [ ]:
# Four-panel overview: DTM | DSM | CHM | AGB
plot_products_agb(store, resolution=10.0, year=YEAR)